In [0]:
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
landing_path = f'/Volumes/bharath_dev_adb_p1/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/'

# Exit gracefully (no failure) if the path or any parent directory doesn't exist, or it holds no files
try:
    landing_files = dbutils.fs.ls(landing_path)
except Exception as e:
    if "FileNotFoundException" in str(e) or "does not exist" in str(e) or "NOT_FOUND" in str(e):
        landing_files = None
    else:
        raise

if not landing_files:
    print(f"No new files are received at {landing_path}. Exiting notebook.")
    dbutils.notebook.exit("0")

print(f"Found {len(landing_files)} item(s) at {landing_path}")

In [0]:
if source_file_format == 'csv':
    landing_df = spark.read.format('csv').option('header', True).option('inferSchema', True).load(landing_path)
else:
    landing_df = spark.read.format('parquet').load(landing_path)

display(landing_df.limit(10))

In [0]:
from pyspark.sql.functions import lit, current_timestamp

updated_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", current_timestamp())

display(updated_df.limit(10))

In [0]:
updated_df.write.mode("append").saveAsTable(f"bharath_dev_adb_p1.bronze.{table_name}")